In [ ]:
'''
I am interested in conducting research on the committor function - comparing a suite of ML algorithms in their ability to estimate 
(as either a classification or regression problem) transitions of trajectories in the Gottwald model. 
To do this, I must train the ML models on data where transitions take place and where they do not. I already have 12 datasets in 
which transitions have been sampled with a rare event algorithm. I now need an equal number of datasets without resampling.
The end of this file should concat all the data - both with transitions and not - which will then serve to train the ML models.
'''

In [1]:
import sys
import math
import matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

In [2]:
'''
load the control run, access state variables evenly spaced to get initial conditions for new trajectories.
Choose iloc[::500] for 1000 trajectories, iloc[::100] for 200 trajectories
'''
def get_initials(start,n):
    
    #control run initial parameterization
    control_run = pd.read_csv('control_run5000_year.csv')
    #remove transient
    control_dropped = control_run.loc[control_run['t']>=10]
    #access every 5th row (time unit)
    control_5th = control_dropped.iloc[::40]
    control_5th.head()
    cols = ['x','y','z','T', 'S']
    x_list, y_list, z_list, temp_list, salt_list = [control_5th[c].tolist()[start:start+n] for c in cols]

    return x_list,y_list,z_list,temp_list,salt_list

In [3]:
#Gottwald model functions

def smoothabs(x, xi=10000):
    """Smooth absolute value function"""
    return x * np.tanh(x * xi)

def gottwald_noice(t, u, p, stochsys=True):
    """Gottwald model functions without sea ice"""
    x, y, z, T, S = u
    if stochsys:
        pvec = p[0] if isinstance(p, list) else p
    else:
        pvec = p
    
    (a, b, mu, epsilon_a, epsilon_f, F0, F1, G0, G1, 
     theta_0, theta_1, sigma_0, sigma_1, x_mean, Delta_mean) = pvec
    
    Delta = y**2 + z**2
    T_surf = theta_0 + theta_1 * (x - x_mean)/(np.sqrt(epsilon_f))
    S_surf = sigma_0 + sigma_1 * (Delta - Delta_mean)/(np.sqrt(epsilon_f))
    
    dx = 1/epsilon_f*(-Delta - a*(x - F0 - F1*T))
    dy = 1/epsilon_f*(x*y - b*x*z - (y - G0 + G1*T))
    dz = 1/epsilon_f*(b*x*y + x*z - z)
    dT = -1/epsilon_a*(T - T_surf) - T - mu*smoothabs(S-T)*T
    dS = S_surf - S - mu*smoothabs(S-T)*S
    
    return np.array([dx, dy, dz, dT, dS])

def param_gwn_default(param_l84=None, sigma_0=0.9): #**kwargs):
    """return parameter values"""
    if param_l84 is not None:
        a, b, F0, G0 = param_l84
    else:
        param_l84 = [0.25, 4., 8., 1.]  # default L84 params [a, b, F0, G0]
        
    x_std = 0.513
    Delta_std = 1.071
    
    x_mean = 1.0147
    Delta_mean = 1.7463  # long-run results from default L84
    
    # Unpack L84 params
    a, b, F0, G0 = param_l84
    
    # Other default model params
    mu = 7.5
    theta_0 = 1.
    # sigma_0 = 0.9  # Stommel params (is varied) 0.926, 0.932
    epsilon_a = 0.34
    epsilon_f = 0.0003  # 0.0083 # timescales
    F1 = 0.1
    G1 = 0.  # coupling params
    perturb_scaling = 0.01  # coupling strength of L84 to Stommel
    
    # Coupling params from L84 run/default
    # theta_1 = min(theta_0, perturb_scaling/x_std)
    # sigma_1 = min(sigma_0, perturb_scaling/Delta_std)

    theta_1=0.0195
    sigma_1=0.00934
    
    return [a, b, mu, epsilon_a, epsilon_f, F0, F1, G0, G1, 
            theta_0, theta_1, sigma_0, sigma_1, x_mean, Delta_mean]
    

def simulate_gottwald_noice(initial_conditions, params, t_span, t_eval=None, stochsys=True):
    """Simulate the Gottwald model without sea ice"""
    sol = solve_ivp(
        lambda t, y: gottwald_noice(t, y, params, stochsys),
        t_span,
        initial_conditions,
        t_eval=t_eval,
        method='RK45',
        rtol=1e-4,
        atol=1e-8
    )
    return sol

#main function for simulating, is called in the loop
def simulate_with_resampling(x_array,y_array,z_array,temp_array,salt_array):
    
    all_runs=[]
    matplotlib.use('Agg')
    
    # Get default Gottwald parameters
    params_gwn = param_gwn_default()
    print(f"Default Gottwald parameters:")
    print(params_gwn)

    # params_gwn_wrapped = [params_gwn]  # Wrap in list for stochsys format
    tmax = 1 # 00
    t_eval = np.linspace(0, tmax, tmax*100)
    amoc=np.zeros((2,2,tmax*100))
    
    # Simulate Gottwald model

    for temp in range(len(temp_array)):
        x0=x_array[temp]
        #print('X0: ',x0,' Trajectory: ',temp)
        y0=y_array[temp]
        z0=z_array[temp]
        T0=temp_array[temp]
        S0=salt_array[temp]
        initial_gwn = [x0, y0, z0, T0, S0]  # [x, y, z, T, S]
            
        sol_gwn = simulate_gottwald_noice(initial_gwn, params_gwn, (0, tmax), 
                                       t_eval=t_eval, stochsys=False)
        data = {
            't': sol_gwn.t,
            'x': sol_gwn.y[0],
            'y': sol_gwn.y[1],
            'z': sol_gwn.y[2],
            'T': sol_gwn.y[3],
            'S': sol_gwn.y[4],
            'x0,y0,z0,T0,S0': f"{x0},{y0},{z0},{T0},{S0}",
            'AMOC': sol_gwn.y[3]-sol_gwn.y[4]
        }

        #print(amoc_df.head())
        
        #Convert this run to a dataframe
        run_df = pd.DataFrame(data)
    
        #Store it
        all_runs.append(run_df)

    #After the loop, combine everything:
    amoc_df = pd.concat(all_runs, ignore_index=True)
    print(amoc_df)
    return amoc_df

In [4]:
run_amoc = []
time=40
ntraj=1000
start=0
for run in range(time):
    print('Run: ',run)
    x_array,y_array,z_array,temp_array,salt_array = get_initials(start,ntraj)
    print(len(x_array))
    print("initials gotten")
    amoc_df = simulate_with_resampling(x_array,y_array,z_array,temp_array,salt_array)
    print("Initial simulation done")
    amoc_df['Run'] = run
    run_amoc.append(amoc_df)

Run:  0
1000
initials gotten
Default Gottwald parameters:
[0.25, 4.0, 7.5, 0.34, 0.0003, 8.0, 0.1, 1.0, 0.0, 1.0, 0.0195, 0.9, 0.00934, 1.0147, 1.7463]
              t         x         y         z         T         S  \
0      0.000000  1.985022  1.506520  0.268272  0.572738  0.413498   
1      0.010101  0.374606 -0.916236  1.620845  0.573811  0.413697   
2      0.020202  1.375792 -0.191505 -0.823606  0.571899  0.413607   
3      0.030303 -0.260126  1.101725 -0.698052  0.571667  0.413909   
4      0.040404  0.185328  1.558537  0.070184  0.569983  0.414004   
...         ...       ...       ...       ...       ...       ...   
99995  0.959596  1.532401  0.382558 -0.169535  0.574851  0.434643   
99996  0.969697  1.183608  1.268528 -0.920092  0.574586  0.434902   
99997  0.979798  1.630356 -0.385541 -1.065521  0.574515  0.435042   
99998  0.989899  0.854428  0.015061  1.243059  0.574945  0.435320   
99999  1.000000  1.982082  1.380040 -0.266625  0.578760  0.435120   

                   

In [5]:
amocs = pd.concat(run_amoc,ignore_index=True)
amocs['Transition']='N'
amocs.to_csv(f"data_{start}.csv")

OSError: [Errno 28] No space left on device